# Telco Customer Churn Analysis & Prediction
**Internship Project | Coding Blocks**

---

## Project Overview

This notebook performs an end-to-end analysis of the IBM Telco Customer Churn dataset with the following goals:

1. **Exploratory Data Analysis (EDA)** – Understand the data distributions and relationships
2. **Data Cleaning** – Handle missing/incorrect values and drop irrelevant columns
3. **Feature Engineering** – Encode categorical variables for ML
4. **Model Training** – Train a Random Forest classifier to predict churn
5. **Model Evaluation** – Assess accuracy, precision, recall, F1-score
6. **Class Balancing & Hyperparameter Tuning** – Improve recall for churners
7. **Customer Segmentation** – Use churn probability to segment customers

---

### Dataset
- **Source:** IBM Sample Dataset – Telco Customer Churn
- **Rows:** 7,043 customers
- **Columns:** 33 features (demographics, services, billing, churn info)
- **Target variable:** `Churn Value` (1 = churned, 0 = stayed)


## Step 1: Import Libraries

We import all required libraries upfront for clarity.
- **pandas / numpy** – data manipulation
- **matplotlib / seaborn** – visualisation
- **scikit-learn** – preprocessing, modelling, evaluation


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    roc_curve,
    roc_auc_score,
    precision_recall_curve,
    auc
)
import xgboost as xgb

# Display settings
pd.set_option('display.max_columns', 50)
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

print("All libraries imported successfully!")


## Step 2: Load the Dataset

We load the Excel file into a pandas DataFrame and take a first look at its structure.


In [ ]:
# Load the dataset
df = pd.read_excel("Telco_customer_churn.xlsx")

# Fix 'Total Charges' – it is stored as text in the source file; convert to numeric
# Entries that cannot be converted (blank cells for new customers) become NaN
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')

print(f"Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns")
df.head()


## Step 3: Initial Data Exploration

### 3.1 – Column Types & Non-Null Counts


In [ ]:
df.info()


### 3.2 – Churn Distribution

How many customers churned vs. stayed?


In [ ]:
churn_counts = df['Churn Label'].value_counts()
print(churn_counts)

# Visual
fig, ax = plt.subplots(figsize=(6, 4))
churn_counts.plot(kind='bar', color=['steelblue', 'tomato'], edgecolor='black', ax=ax)
ax.set_title('Overall Churn Distribution')
ax.set_xlabel('Churn Label')
ax.set_ylabel('Number of Customers')
ax.set_xticklabels(['No Churn', 'Churned'], rotation=0)
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}', (p.get_x() + 0.1, p.get_height() + 30))
plt.tight_layout()
plt.show()

churn_rate = churn_counts['Yes'] / len(df) * 100
print(f"\nChurn Rate: {churn_rate:.1f}%")


## Step 4: Exploratory Data Analysis (EDA)

We explore the key features that are likely to influence churn.

### 4.1 – Tenure Distribution
Customer tenure (in months) tells us how long a customer has been with the company.


In [ ]:
print(f"Min tenure: {df['Tenure Months'].min()} months")
print(f"Max tenure: {df['Tenure Months'].max()} months")
print(f"Mean tenure: {df['Tenure Months'].mean():.1f} months")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
sns.histplot(df['Tenure Months'], bins=30, kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Distribution of Tenure Months')
axes[0].set_xlabel('Tenure Months')
axes[0].set_ylabel('Customer Count')

# Box plot by churn
sns.boxplot(x='Churn Label', y='Tenure Months', data=df, ax=axes[1],
            palette={'No': 'steelblue', 'Yes': 'tomato'})
axes[1].set_title('Tenure Months by Churn Status')
axes[1].set_xlabel('Churn Label')
axes[1].set_ylabel('Tenure Months')

plt.tight_layout()
plt.show()


### 4.2 – Monthly Charges Distribution


In [ ]:
print(df['Monthly Charges'].describe())
print()
print("Quartiles by Churn Status:")
for label in ['Yes', 'No']:
    q = df[df['Churn Label'] == label]['Monthly Charges'].quantile([0.25, 0.5, 0.75])
    print(f"  Churn={label}: Q1={q[0.25]:.1f}, Median={q[0.5]:.1f}, Q3={q[0.75]:.1f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df['Monthly Charges'], bins=30, kde=True, ax=axes[0], color='mediumseagreen')
axes[0].set_title('Distribution of Monthly Charges')
axes[0].set_xlabel('Monthly Charges ($)')
axes[0].set_ylabel('Customer Count')

sns.boxplot(x='Churn Label', y='Monthly Charges', data=df, ax=axes[1],
            palette={'No': 'steelblue', 'Yes': 'tomato'})
axes[1].set_title('Monthly Charges by Churn Status')
axes[1].set_xlabel('Churn Label')
axes[1].set_ylabel('Monthly Charges ($)')

plt.tight_layout()
plt.show()


### 4.3 – Categorical Feature Analysis

We visualise how Contract type, Internet Service, Payment Method, and Tech Support relate to churn.


In [ ]:
cat_features = ['Contract', 'Internet Service', 'Payment Method', 'Tech Support']

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for i, feature in enumerate(cat_features):
    sns.countplot(
        x=feature, hue='Churn Label', data=df, ax=axes[i],
        palette={'No': 'steelblue', 'Yes': 'tomato'}
    )
    axes[i].set_title(f'Churn by {feature}')
    axes[i].set_xlabel(feature)
    axes[i].set_ylabel('Count')
    axes[i].tick_params(axis='x', rotation=20)
    axes[i].legend(title='Churn', loc='upper right')

plt.suptitle('Categorical Features vs Churn', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


### 4.4 – Average Tenure by Churn Status


In [ ]:
avg_tenure = df.groupby('Churn Label')['Tenure Months'].mean()
print("Average Tenure by Churn Status:")
print(avg_tenure.round(2))


### 4.5 – Correlation Heatmap (Numerical Features)


In [ ]:
numerical_cols = ['Tenure Months', 'Monthly Charges', 'Total Charges', 'Churn Score', 'CLTV']
corr = df[numerical_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', square=True, linewidths=0.5)
plt.title('Correlation Matrix – Numerical Features')
plt.tight_layout()
plt.show()


### 4.6 – Contract Type vs Churn Rate (Cross-tab)


In [ ]:
contract_churn_abs = pd.crosstab(df['Contract'], df['Churn Label'])
contract_churn_pct = pd.crosstab(df['Contract'], df['Churn Label'], normalize='index').round(3) * 100

print("Absolute counts:")
print(contract_churn_abs)
print("\nChurn rate (%) by contract type:")
print(contract_churn_pct)


## Step 5: Data Cleaning

### 5.1 – Handle Missing Values in 'Total Charges'

`Total Charges` is missing for customers with 0 tenure months (new customers who haven't been billed yet).
We fill those with 0.


In [ ]:
# How many NaN values and what is their tenure?
null_tc = df[df['Total Charges'].isnull()]
print(f"Rows with missing Total Charges: {len(null_tc)}")
print("Their Tenure Months values:")
print(null_tc['Tenure Months'].value_counts())

# Fill with 0 – they haven't been charged yet
df['Total Charges'] = df['Total Charges'].fillna(0)

print(f"\nRemaining nulls in Total Charges: {df['Total Charges'].isnull().sum()}")


### 5.2 – Drop Irrelevant Columns

The following columns are not useful for churn prediction:
- **Identifiers**: CustomerID, Count
- **Geography** (too granular): Country, State, City, Zip Code, Lat Long, Latitude, Longitude
- **Target leakage**: Churn Label, Churn Score, CLTV, Churn Reason
  *(these contain information derived from or related to the churn event itself)*


In [ ]:
drop_columns = [
    'CustomerID', 'Count', 'Country', 'State', 'City',
    'Zip Code', 'Lat Long', 'Latitude', 'Longitude',
    'Churn Label',   # text version of target – redundant with Churn Value
    'Churn Score',   # post-hoc churn score – data leakage
    'CLTV',          # customer lifetime value – derived metric
    'Churn Reason'   # available only after churn – data leakage
]

df_clean = df.drop(columns=drop_columns)
print(f"Remaining shape: {df_clean.shape}")
print("Remaining columns:", df_clean.columns.tolist())


## Step 6: Feature Engineering

We use **one-hot encoding** (`pd.get_dummies`) to convert all categorical (text) columns into numeric 0/1 columns that the ML model can use.

`drop_first=True` removes one dummy per category to avoid multicollinearity (the **dummy variable trap**).


In [ ]:
df_encoded = pd.get_dummies(df_clean, drop_first=True)

# Separate features (X) and target (Y)
X = df_encoded.drop('Churn Value', axis=1)
Y = df_encoded['Churn Value']

print(f"Feature matrix shape : {X.shape}")
print(f"Target vector shape  : {Y.shape}")
print(f"\nChurn rate in cleaned dataset: {Y.mean()*100:.1f}%")
print(f"\nFeature names ({len(X.columns)}):")
for col in X.columns:
    print(f"  {col}")


## Step 7: Train / Test Split

We split the data 80/20 with `random_state=42` for reproducibility.
- **80% training set** – used to fit the model
- **20% test set** – held out to evaluate real-world performance


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, Y,
    test_size=0.2,
    random_state=42
)

print(f"Training set  : {X_train.shape[0]} samples")
print(f"Test set      : {X_test.shape[0]} samples")
print(f"\nChurn rate in training set : {y_train.mean()*100:.1f}%")
print(f"Churn rate in test set     : {y_test.mean()*100:.1f}%")


## Step 8: Baseline Random Forest Model

**Random Forest** is an ensemble of decision trees that averages their predictions.  
It handles mixed feature types well and gives us feature importance for free.

Hyperparameters used here:
- `n_estimators=100` – 100 trees in the forest
- `random_state=42` – reproducible results


In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
print("Baseline model trained!")


### 8.1 – Evaluate Baseline Model


In [ ]:
y_pred = rf_model.predict(X_test)

print("=" * 50)
print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print("=" * 50)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churned']))

# Confusion matrix visual
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Churn', 'Churned'])
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Baseline Model – Confusion Matrix')
plt.tight_layout()
plt.show()


## Step 9: Class-Balanced Model

The dataset is imbalanced (~26% churn). The baseline model may be biased towards predicting "No Churn".

We use `class_weight='balanced'` which automatically adjusts weights inversely proportional to class frequencies, penalising the model more for missing a churner.


In [ ]:
rf_balanced = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight='balanced'
)
rf_balanced.fit(X_train, y_train)
y_pred_balanced = rf_balanced.predict(X_test)

print("=" * 50)
print(f"Balanced Model Accuracy : {accuracy_score(y_test, y_pred_balanced):.4f}")
print("=" * 50)

print("\nClassification Report (Balanced):")
print(classification_report(y_test, y_pred_balanced, target_names=['No Churn', 'Churned']))

cm_balanced = confusion_matrix(y_test, y_pred_balanced)
disp2 = ConfusionMatrixDisplay(confusion_matrix=cm_balanced, display_labels=['No Churn', 'Churned'])
fig, ax = plt.subplots(figsize=(6, 5))
disp2.plot(ax=ax, cmap='Greens', colorbar=False)
ax.set_title('Balanced Model – Confusion Matrix')
plt.tight_layout()
plt.show()


## Step 10: Hyperparameter Tuning

We manually tune:
- `n_estimators=300` – more trees for better stability
- `max_depth=10` – limit tree depth to reduce overfitting

Combined with `class_weight='balanced'` to keep churn recall high.


In [ ]:
rf_tuned = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    random_state=42,
    class_weight='balanced'
)
rf_tuned.fit(X_train, y_train)
y_pred_tuned = rf_tuned.predict(X_test)

print("=" * 50)
print(f"Tuned Model Accuracy : {accuracy_score(y_test, y_pred_tuned):.4f}")
print("=" * 50)

print("\nClassification Report (Tuned):")
print(classification_report(y_test, y_pred_tuned, target_names=['No Churn', 'Churned']))

cm_tuned = confusion_matrix(y_test, y_pred_tuned)
disp3 = ConfusionMatrixDisplay(confusion_matrix=cm_tuned, display_labels=['No Churn', 'Churned'])
fig, ax = plt.subplots(figsize=(6, 5))
disp3.plot(ax=ax, cmap='Oranges', colorbar=False)
ax.set_title('Tuned Model – Confusion Matrix')
plt.tight_layout()
plt.show()


### 10.1 – ROC Curve and AUC

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

# Get probabilities for the positive class (churned)
y_prob_tuned = rf_tuned.predict_proba(X_test)[:, 1]

# Calculate ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_prob_tuned)

# Calculate AUC
auc_tuned = roc_auc_score(y_test, y_prob_tuned)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {auc_tuned:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

print(f"AUC for Tuned Model: {auc_tuned:.4f}")

### 10.2 – Precision-Recall Curve

In [ ]:
from sklearn.metrics import precision_recall_curve, auc

# Get probabilities for the positive class (churned)
y_prob_tuned = rf_tuned.predict_proba(X_test)[:, 1]

# Calculate Precision-Recall curve
precision, recall, thresholds = precision_recall_curve(y_test, y_prob_tuned)

# Calculate Area Under the Precision-Recall Curve (AUPRC)
auprc_tuned = auc(recall, precision)

plt.figure(figsize=(8, 6))
plt.plot(recall, precision, color='purple', lw=2, label=f'Precision-Recall curve (AUPRC = {auprc_tuned:.2f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc='lower left')
plt.grid(True)
plt.show()

print(f"AUPRC for Tuned Model: {auprc_tuned:.4f}")

## Step 11: XGBoost Model – Improved Recall

The dataset has ~26% churn (class imbalance). Random Forest corrects for this via `class_weight='balanced'`,
but XGBoost handles it more directly through **`scale_pos_weight`**.

- `scale_pos_weight = (# non-churners) / (# churners)` — tells XGBoost to penalise missing a churner more during gradient updates
- This is more effective than post-hoc reweighting because it influences the *learning process itself*
- Result: higher recall on the minority class (churners) — which is the business-critical metric


In [ ]:

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    random_state=42
)

xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)

print("=" * 50)
print(f"XGBoost Accuracy : {accuracy_score(y_test, y_pred_xgb):.4f}")
print("=" * 50)

print("\nClassification Report (XGBoost):")
print(classification_report(y_test, y_pred_xgb, target_names=['No Churn', 'Churned']))

# Confusion matrix
cm_xgb = confusion_matrix(y_test, y_pred_xgb)
disp_xgb = ConfusionMatrixDisplay(confusion_matrix=cm_xgb, display_labels=['No Churn', 'Churned'])
fig, ax = plt.subplots(figsize=(6, 5))
disp_xgb.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('XGBoost – Confusion Matrix')
plt.tight_layout()
plt.show()


### 11.1 – ROC Curve Comparison (RF vs XGBoost)


In [ ]:
# Compare ROC curves of Tuned RF vs XGBoost
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]
y_prob_tuned = rf_tuned.predict_proba(X_test)[:, 1]

fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_tuned)
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_prob_xgb)
auc_rf = roc_auc_score(y_test, y_prob_tuned)
auc_xgb = roc_auc_score(y_test, y_prob_xgb)

plt.figure(figsize=(8, 6))
plt.plot(fpr_rf, tpr_rf, color='darkorange', lw=2, label=f'Tuned RF (AUC = {auc_rf:.2f})')
plt.plot(fpr_xgb, tpr_xgb, color='steelblue', lw=2, label=f'XGBoost (AUC = {auc_xgb:.2f})')
plt.plot([0, 1], [0, 1], color='grey', lw=1.5, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve – Tuned RF vs XGBoost')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

print(f"Tuned RF AUC : {auc_rf:.4f}")
print(f"XGBoost AUC  : {auc_xgb:.4f}")


## Step 12: Feature Importance

Random Forests provide importance scores based on impurity reduction across all trees.  
XGBoost uses gain-based importance — how much each feature contributes to the boosting steps.  
Higher = more predictive in both cases.


In [ ]:
# RF Feature Importance
feature_importance_rf = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_tuned.feature_importances_
}).sort_values(by='Importance', ascending=False).reset_index(drop=True)

print("Top 15 Most Important Features (Tuned RF):")
print(feature_importance_rf.head(15).to_string(index=False))

plt.figure(figsize=(10, 6))
sns.barplot(
    data=feature_importance_rf.head(15),
    x='Importance', y='Feature',
    palette='viridis'
)
plt.title('Top 15 Feature Importances – Tuned Random Forest')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()


### 12.2 – XGBoost Feature Importance


In [ ]:
# XGBoost Feature Importance
feature_importance_xgb = pd.DataFrame({
    'Feature': X.columns,
    'Importance': xgb_model.feature_importances_
}).sort_values(by='Importance', ascending=False).reset_index(drop=True)

print("Top 15 Most Important Features (XGBoost):")
print(feature_importance_xgb.head(15).to_string(index=False))

plt.figure(figsize=(10, 6))
sns.barplot(
    data=feature_importance_xgb.head(15),
    x='Importance', y='Feature',
    palette='rocket'
)
plt.title('Top 15 Feature Importances – XGBoost')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()


## Step 13: Cross-Validation

5-fold cross-validation gives a more reliable estimate of model performance than a single train/test split.  
We evaluate both **accuracy** and **recall** for the Tuned RF and XGBoost models.


In [ ]:
# Tuned RF Cross-Validation
final_rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    random_state=42,
    class_weight='balanced'
)

cv_accuracy_rf = cross_val_score(final_rf, X, Y, cv=5, scoring='accuracy')
cv_recall_rf   = cross_val_score(final_rf, X, Y, cv=5, scoring='recall')

print("Cross-Validation – Tuned Random Forest (5-fold):")
print(f"  Accuracy : {np.round(cv_accuracy_rf, 4)} | Mean={cv_accuracy_rf.mean():.4f}")
print(f"  Recall   : {np.round(cv_recall_rf, 4)} | Mean={cv_recall_rf.mean():.4f}")

print()

# XGBoost Cross-Validation
cv_accuracy_xgb = cross_val_score(xgb_model, X, Y, cv=5, scoring='accuracy')
cv_recall_xgb   = cross_val_score(xgb_model, X, Y, cv=5, scoring='recall')

print("Cross-Validation – XGBoost (5-fold):")
print(f"  Accuracy : {np.round(cv_accuracy_xgb, 4)} | Mean={cv_accuracy_xgb.mean():.4f}")
print(f"  Recall   : {np.round(cv_recall_xgb, 4)} | Mean={cv_recall_xgb.mean():.4f}")


## Step 14: Customer Segmentation by Churn Probability

We use **XGBoost's** predicted churn probability (our best recall model) to build a **risk-based customer segmentation**:

| Segment | Churn Probability | Business Action |
|---------|-------------------|-----------------|
| Low Risk | < 30% | Maintain relationship |
| Medium Risk | 30% – 60% | Proactive engagement |
| High Risk | > 60% | Urgent retention campaign |


In [ ]:
# Use XGBoost probabilities for segmentation (best recall model)
y_prob_all = xgb_model.predict_proba(X)
churn_prob = y_prob_all[:, 1]

segmentation_data = pd.DataFrame({
    'Tenure Months': X['Tenure Months'].values,
    'Monthly Charges': X['Monthly Charges'].values,
    'Total Charges': X['Total Charges'].values,
    'Churn Probability': churn_prob
})

def assign_segment(prob):
    if prob < 0.30:
        return 'Low Risk'
    elif prob < 0.60:
        return 'Medium Risk'
    else:
        return 'High Risk'

segmentation_data['Risk Segment'] = segmentation_data['Churn Probability'].apply(assign_segment)

print("Segmentation Summary:")
print(segmentation_data['Risk Segment'].value_counts())
print(f"\nSample (top 10 rows):")
segmentation_data.head(10)


### 14.1 – Segment Size Distribution


In [ ]:
seg_counts = segmentation_data['Risk Segment'].value_counts()
colors = {'Low Risk': 'steelblue', 'Medium Risk': 'gold', 'High Risk': 'tomato'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

seg_counts.plot(kind='bar',
                color=[colors[s] for s in seg_counts.index],
                edgecolor='black', ax=axes[0])
axes[0].set_title('Customer Count by Risk Segment')
axes[0].set_xlabel('Risk Segment')
axes[0].set_ylabel('Number of Customers')
axes[0].set_xticklabels(seg_counts.index, rotation=0)
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}',
                     (p.get_x() + 0.08, p.get_height() + 30))

axes[1].pie(seg_counts,
            labels=seg_counts.index,
            autopct='%1.1f%%',
            colors=[colors[s] for s in seg_counts.index],
            startangle=140)
axes[1].set_title('Risk Segment Distribution')

plt.tight_layout()
plt.show()


### 14.2 – Monthly Charges vs Tenure by Risk Segment


In [ ]:
plt.figure(figsize=(10, 6))
for segment, color in [('Low Risk', 'steelblue'), ('Medium Risk', 'gold'), ('High Risk', 'tomato')]:
    subset = segmentation_data[segmentation_data['Risk Segment'] == segment]
    plt.scatter(
        subset['Tenure Months'],
        subset['Monthly Charges'],
        c=color, label=segment, alpha=0.5, s=20
    )

plt.xlabel('Tenure Months')
plt.ylabel('Monthly Charges ($)')
plt.title('Customer Segments: Tenure vs Monthly Charges')
plt.legend(title='Risk Segment')
plt.tight_layout()
plt.show()


### 14.3 – Segment Profile Summary


In [ ]:
profile = segmentation_data.groupby('Risk Segment')[
    ['Tenure Months', 'Monthly Charges', 'Total Charges', 'Churn Probability']
].mean().round(2)

print("Average Profile per Risk Segment:")
print(profile)


## Step 15: Summary & Key Findings

### EDA Insights
- **Tenure**: Customers who churn have significantly shorter tenure (mean ~18 months vs ~38 months for retained customers).
- **Monthly Charges**: Churners pay higher monthly charges on average (~$74 vs ~$61).
- **Contract Type**: Month-to-month contracts have the highest churn rate (~43%). Two-year contracts have the lowest (~3%).
- **Payment Method**: Electronic check users churn the most; auto-pay customers churn the least.
- **Tech Support**: Customers without tech support churn at nearly double the rate.

### Model Performance Summary
| Model | Accuracy | Churner Recall |
|-------|----------|----------------|
| Baseline RF | ~79% | ~48% |
| Balanced RF | ~77% | ~72% |
| Tuned RF | ~78% | ~75% |
| **XGBoost** | **~77%** | **~78%** |

> **XGBoost** is our final model — `scale_pos_weight` directly adjusts gradient updates during training,
> giving it the highest recall on churners (78%) while keeping overall accuracy stable.

### Customer Segments
| Segment | Churn Prob | Recommended Action |
|---------|-----------|-------------------|
| Low Risk | < 30% | Loyalty rewards; upsell opportunities |
| Medium Risk | 30–60% | Proactive outreach; service improvement |
| High Risk | > 60% | Urgent retention offers; personal contact |

### Business Recommendations
1. **Incentivise long-term contracts** – reduce month-to-month plans or offer discounts for annual commitment.
2. **Promote auto-pay** – customers on electronic check churn more; nudge them to auto-pay.
3. **Bundle tech support** – especially for high monthly-charge customers who lack it.
4. **Target high-risk segment** with personalised retention campaigns.
